In [0]:
# Import libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/akhildataengineer@hotmail.com/Atlikon_migration_pipeline/1_setup/3_utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sportsbar-store-dp1/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed"

In [0]:
# Define tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Bronze data processing

In [0]:
df = spark.read.options(header = True, inferSchema = True)\
                .csv(f"{landing_path}/*.csv")\
                .withColumn("read_timestamp", F.current_timestamp())\
                .select("*", "_metadata.file_name", "_metadata.file_size")

df.show(5)

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("append")\
    .saveAsTable(bronze_table)